In [ ]:
# =============================================================================
# 01_preprocessing.ipynb
# Load, align, mask, and QA all GEE seasonal exports.
#
# Inputs : GeoTIFFs in /content/drive/MyDrive/Soil_Moisture/ (from GEE)
# Outputs: Clean aligned NetCDF stacks in /content/drive/MyDrive/mkfe_processed/
# Master grid: EPSG:21037, 30 m, snapped to Landsat grid
#
# Pipeline:
#   1.  Discover files
#   2.  Build master grid from a reference Landsat file
#   3.  Load Landsat (NDVI, LST) + NOBS
#   4.  Load SRTM (already 30 m)
#   5.  Load CHIRPS, ERA5 (SM, T), FIRMS — resample to master grid
#   6.  Apply QA (NOBS filter, -9999 → NaN)
#   7.  Save per-variable NetCDF stacks: dims (time, y, x)
# =============================================================================

import os, re, glob, json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.enums import Resampling
from tqdm.auto import tqdm

# ---------------- Config ----------------
DRIVE_IN  = Path('/content/drive/MyDrive/Soil_Moisture')
DRIVE_OUT = Path('/content/drive/MyDrive/mkfe_processed')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

SEASONS   = ['JF', 'MAM', 'JJAS', 'OND']
YEARS     = list(range(1995, 2026))
NODATA    = -9999

# Master grid
CRS       = 'EPSG:21037'
RES       = 30                # metres
NOBS_MIN  = 2                 # minimum clear observations per pixel per season

# Dataset -> (variable name, native resampling method)
DATASETS = {
    'NDVI':       ('NDVI',       None),        # already 30 m
    'LST':        ('LST',        None),        # already 30 m
    'NOBS_NDVI':  ('NOBS_NDVI',  None),
    'NOBS_LST':   ('NOBS_LST',   None),
    'SM_L1':      ('SM_L1',      Resampling.bilinear),
    'SM_L2':      ('SM_L2',      Resampling.bilinear),
    'SM_L3':      ('SM_L3',      Resampling.bilinear),
    'SM_L4':      ('SM_L4',      Resampling.bilinear),
    'T2M':        ('T2M',        Resampling.bilinear),
    'T2M_MAX':    ('T2M_MAX',    Resampling.bilinear),
    'T2M_MIN':    ('T2M_MIN',    Resampling.bilinear),
    'PRECIP':     ('PRECIP',     Resampling.bilinear),
    'FIRE_DAYS':  ('FIRE_DAYS',  Resampling.nearest),
}

In [ ]:
# =============================================================================
# Discover all GeoTIFFs, parse <VAR>_<SEASON>_<YEAR>.tif filenames.
# =============================================================================

pattern = re.compile(r'^(?P<var>[A-Z0-9_]+)_(?P<season>JF|MAM|JJAS|OND)_(?P<year>\d{4})\.tif$')

records = []
for f in DRIVE_IN.glob('*.tif'):
    m = pattern.match(f.name)
    if not m:
        continue
    records.append({
        'path':   f,
        'var':    m.group('var'),
        'season': m.group('season'),
        'year':   int(m.group('year')),
    })

manifest = pd.DataFrame(records)
print(f"Discovered {len(manifest)} files.")
print(manifest.groupby('var').size().sort_values(ascending=False))

# Save manifest for reference
manifest.to_csv(DRIVE_OUT / 'manifest.csv', index=False)

In [ ]:
# =============================================================================
# Use one Landsat file as the spatial reference. All other rasters will be
# reprojected / resampled onto this exact grid.
# =============================================================================

# Pick a well-populated reference: NDVI_JF for a mid-period year
ref_row = manifest[(manifest['var'] == 'NDVI') & (manifest['season'] == 'JF') &
                   (manifest['year'] == 2010)].iloc[0]

ref = rxr.open_rasterio(ref_row['path'], masked=True).squeeze()
ref = ref.rio.write_crs(CRS, inplace=False)
ref = ref.rio.write_nodata(np.nan, inplace=False)

# Reproject to master CRS if needed (should already be 21037)
if str(ref.rio.crs) != CRS:
    ref = ref.rio.reproject(CRS, resolution=RES, resampling=Resampling.nearest)

master_transform = ref.rio.transform()
master_shape     = ref.shape
master_x         = ref.x.values
master_y         = ref.y.values

print(f"Master grid: {master_shape[1]} cols × {master_shape[0]} rows")
print(f"Transform: {master_transform}")
print(f"CRS: {ref.rio.crs}")
print(f"Resolution: {ref.rio.resolution()}")

In [ ]:
# =============================================================================
# Load any GeoTIFF, resample to master grid, convert nodata to NaN.
# =============================================================================

def load_aligned(path, resampling=None):
    """Return 2D numpy array aligned to master grid, with NaN for nodata."""
    da = rxr.open_rasterio(path, masked=True).squeeze()

    # Ensure CRS is set
    if da.rio.crs is None:
        da = da.rio.write_crs(CRS, inplace=False)

    # Reproject/resample onto master grid if needed
    if da.shape != master_shape or str(da.rio.crs) != CRS:
        if resampling is None:
            resampling = Resampling.bilinear
        da = da.rio.reproject(
            CRS,
            shape=master_shape,
            transform=master_transform,
            resampling=resampling,
        )

    arr = da.values.astype('float32')

    # Mask nodata
    arr[arr == NODATA] = np.nan
    arr[~np.isfinite(arr)] = np.nan
    return arr

In [ ]:
# =============================================================================
# For each variable, build a (time, y, x) array. time index = (year, season).
# Missing files are simply skipped — the time dimension will be ragged.
# To make a clean stack, we build a full (year × season) index and fill with NaN
# where data is absent.
# =============================================================================

def build_variable_stack(var_name, resampling=None):
    """
    Returns xr.DataArray with dims (time, y, x), where time is a MultiIndex
    of (year, season). Missing (year, season) combos are NaN slabs.
    """
    rows = manifest[manifest['var'] == var_name]

    # Full index
    full_index = pd.MultiIndex.from_product(
        [YEARS, SEASONS], names=['year', 'season']
    )

    # Pre-allocate with NaN
    data = np.full((len(full_index),) + master_shape, np.nan, dtype='float32')

    for _, r in tqdm(rows.iterrows(), total=len(rows), desc=var_name):
        idx = full_index.get_loc((r['year'], r['season']))
        try:
            data[idx] = load_aligned(r['path'], resampling=resampling)
        except Exception as e:
            print(f"  ✗ {r['path'].name}: {e}")

    da = xr.DataArray(
        data,
        dims=('time', 'y', 'x'),
        coords={
            'time': full_index,
            'y':    master_y,
            'x':    master_x,
        },
        name=var_name,
        attrs={
            'crs':        CRS,
            'res':        RES,
            'nodata':     'NaN',
            'units':      'see source',
            'source':     'GEE seasonal export',
        },
    )
    da = da.rio.write_crs(CRS, inplace=False)
    da = da.rio.write_transform(master_transform, inplace=False)
    return da

In [ ]:
# =============================================================================
# Build and save each variable's stack.
# =============================================================================

for var, (_, resampling) in DATASETS.items():
    if var not in manifest['var'].unique():
        print(f"⚠ {var}: no files found, skipping.")
        continue

    print(f"\n▶ Building {var} ...")
    stack = build_variable_stack(var, resampling=resampling)

    out_path = DRIVE_OUT / f'{var}_stack.nc'
    stack.to_netcdf(out_path, engine='netcdf4')
    print(f"  → saved {out_path.name}  shape={stack.shape}")

In [ ]:
# =============================================================================
# Apply per-pixel QA using NOBS_NDVI / NOBS_LST.
# Where NOBS < NOBS_MIN, set NDVI / LST to NaN.
# =============================================================================

def qa_mask(target_var, nobs_var):
    tgt  = xr.open_dataarray(DRIVE_OUT / f'{target_var}_stack.nc')
    nobs = xr.open_dataarray(DRIVE_OUT / f'{nobs_var}_stack.nc')

    # Align
    nobs = nobs.reindex_like(tgt)

    masked = tgt.where(nobs >= NOBS_MIN)
    masked.attrs = tgt.attrs.copy()
    masked.attrs['qa'] = f'NOBS >= {NOBS_MIN} applied'

    out_path = DRIVE_OUT / f'{target_var}_stack_qa.nc'
    masked.to_netcdf(out_path, engine='netcdf4')
    print(f"✓ {target_var}: QA applied, saved to {out_path.name}")
    return masked

ndvi = qa_mask('NDVI', 'NOBS_NDVI')
lst  = qa_mask('LST',  'NOBS_LST')

In [ ]:
# =============================================================================
# SRTM is a single static file, no seasonal stack needed.
# =============================================================================

srtm = rxr.open_rasterio(DRIVE_IN / 'SRTM_ELEVATION.tif', masked=True).squeeze()
if srtm.shape != master_shape:
    srtm = srtm.rio.reproject(
        CRS,
        shape=master_shape,
        transform=master_transform,
        resampling=Resampling.bilinear,
    )
srtm = srtm.where(srtm != NODATA)
srtm = srtm.astype('float32')

srtm.to_netcdf(DRIVE_OUT / 'SRTM_stack.nc', engine='netcdf4')
print(f"✓ SRTM saved. shape={srtm.shape}, range={float(srtm.min()):.0f}–{float(srtm.max()):.0f} m")

In [ ]:
# =============================================================================
# Summarise what was produced: shapes, valid fraction, value ranges.
# =============================================================================

summary = []
for var in DATASETS.keys():
    qa_path  = DRIVE_OUT / f'{var}_stack_qa.nc'
    raw_path = DRIVE_OUT / f'{var}_stack.nc'
    path = qa_path if qa_path.exists() else raw_path
    if not path.exists():
        continue

    da = xr.open_dataarray(path)
    valid = float(np.isfinite(da.values).sum()) / da.values.size
    summary.append({
        'variable': var,
        'shape':    da.shape,
        'valid_%':  round(valid * 100, 1),
        'min':      float(np.nanmin(da.values)),
        'max':      float(np.nanmax(da.values)),
        'file':     path.name,
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(DRIVE_OUT / 'preprocessing_summary.csv', index=False)

In [ ]:
# =============================================================================
# Quick visual check: NDVI for JF 2010 and LST for JJAS 2010.
# =============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ndvi = xr.open_dataarray(DRIVE_OUT / 'NDVI_stack_qa.nc')
lst  = xr.open_dataarray(DRIVE_OUT / 'LST_stack_qa.nc')

sel_ndvi = ndvi.sel(time=('2010', 'JF')).squeeze()
sel_lst  = lst.sel(time=('2010', 'JJAS')).squeeze()

axes[0].imshow(sel_ndvi, cmap='YlGn', vmin=0, vmax=0.9)
axes[0].set_title('NDVI — JF 2010')
axes[0].axis('off')

axes[1].imshow(sel_lst, cmap='inferno')
axes[1].set_title('LST (K) — JJAS 2010')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(DRIVE_OUT / 'preview.png', dpi=120)
plt.show()